# Run the Stage 1 GHI pipeline (orchestrator)

[!] writing to `*_v2` tables so his originals are never touched. [!] 

```
build_structured_data  →  build_split_days  →  build_ghi_model  →  build_all_uncurtailedpv
        (_v2)                   (_v2)                (_v2)                  (_v2)
```
**Cost note:** the structured_data build scans `ts` (billions of rows). Run the
**tiny test slice first** (one month, one site-part) and confirm it works before
scaling to the full year. Athena bills by data scanned.

In [2]:
# Bootstrap: paths + imports
import sys
import pathlib
import importlib

ROOT = pathlib.Path.cwd().parents[1]
SHARED = ROOT / "shared"
STAGE1 = ROOT / "data_calc_write" / "stage1_ghi_pipeline"

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
sys.path.insert(0, str(STAGE1))

from aws_config import aq            # existing Athena helper
from ciccada_config import SAI       # 'solar_analytics_iceberg'

import build_structured_data   as b1
import build_split_days        as b2
import build_ghi_model         as b3
import build_mape_quality_gate as b3b
import build_all_uncurtailedpv as b4

b1 = importlib.reload(b1)
b2 = importlib.reload(b2)
b3 = importlib.reload(b3)
b3b = importlib.reload(b3b)
b4 = importlib.reload(b4)

'''
# Current analytical choice, excluding flex-export-detected sites from the MAPE quality gate.
# remove only flex_export_detected=True.
FLEX_SELECTION = "exclude"

STAGE1_OPTIONS = dict(
    normalization_basis="s_99",
    voltage_aggregation="avg",
    flex_selection=FLEX_SELECTION,
)

MAPE_CSV = "mape_under50_sites.csv"
MAPE_METRICS_CSV = "mape_metrics_sites.csv"

print("Target tables (note the _v2 suffix. Original results untouched):")
print(" ", b1.TARGET)
print(" ", b2.TARGET)
print(" ", b3.TARGET)
print(" ", b4.TARGET)

'''

# Replication of MAPE quality gate with flex-export-detected sites included (milestone 3 report).
# This branch includes sites where flex_export_detected=True.
FLEX_SELECTION = "include"

SD_TARGET = "structured_data_v2_flex_included"
SPLIT_TARGET = "split_days_v2_flex_included"
MODEL_TARGET = "pv_ghi_norm_model_v2_flex_included"
UNCURTAILED_TARGET = "all_uncurtailedpv_v2_flex_included"

MAPE_CSV = "mape_under50_sites_flex_included.csv"
MAPE_METRICS_CSV = "mape_metrics_flex_included.csv"

STAGE1_OPTIONS = dict(
    normalization_basis="s_99",
    voltage_aggregation="avg",
    flex_selection=FLEX_SELECTION,
    target=SD_TARGET,
)

## Step 1. Structured_data_v2

In [3]:
# 1a. Create the empty table (safe: drops & recreates only the _v2 table)
print(b1.create_table(
    aq,
    database=SAI,
    target=SD_TARGET,
))

Created empty structured_data_v2_flex_included


In [5]:
# 1b. TEST SLICE FIRST. 
# One month, one of 8 site-parts.
# Confirm this completes and validate() looks sane BEFORE the full run.
b1.run_slice(
    aq,
    database=SAI,
    year=2024,
    months=[1],
    n_parts=8,
    parts=[0],
    **STAGE1_OPTIONS
)

loaded year=2024 month=1 part=0/8


['loaded year=2024 month=1 part=0/8']

In [6]:
# 1c. Validate the test slice
b1.validate(aq, database=SAI, target=SD_TARGET)

Distinct sites: 1,195
Voltage / P_norm sanity:
 v_min  v_avg  v_max  p_norm_avg
  22.3  240.8  262.6       0.368


(      n
 0  1195,
    v_min  v_avg  v_max  p_norm_avg
 0   22.3  240.8  262.6       0.368)

In [7]:
# 1d. FULL RUN. All months, all 16 site-parts, 2024 + 2025.
#     Uses run_resilient: paces the queries, retries on "exhausted resources",
#     and halves any slice that still fails. A failed Athena INSERT writes
#     nothing, so retries and sub-splits cannot duplicate rows.
#     create_table first: the two failed runs left partial data behind.
print(b1.create_table(aq, database=SAI, target=SD_TARGET))

Created empty structured_data_v2_flex_included


In [ ]:
done24, failed24 = b1.run_resilient(
    b1,
    aq,
    SAI,
    year=2024,
    months=range(1, 13),
    n_parts=8,
    pause=1,
    **STAGE1_OPTIONS,
)

loaded year=2024 month=1 part=0/8
loaded year=2024 month=1 part=1/8
loaded year=2024 month=1 part=2/8
loaded year=2024 month=1 part=3/8
loaded year=2024 month=1 part=4/8


In [ ]:
done25, failed25 = b1.run_resilient(
    b1,
    aq,
    SAI,
    year=2025,
    months=range(1, 13),
    n_parts=8,
    pause=1,
    **STAGE1_OPTIONS,
)

loaded year=2025 month=1 part=0/8
loaded year=2025 month=1 part=1/8
loaded year=2025 month=1 part=2/8
loaded year=2025 month=1 part=3/8
loaded year=2025 month=1 part=4/8
loaded year=2025 month=1 part=5/8
loaded year=2025 month=1 part=6/8
loaded year=2025 month=1 part=7/8
loaded year=2025 month=2 part=0/8
loaded year=2025 month=2 part=1/8
loaded year=2025 month=2 part=2/8
loaded year=2025 month=2 part=3/8
loaded year=2025 month=2 part=4/8
loaded year=2025 month=2 part=5/8
loaded year=2025 month=2 part=6/8
loaded year=2025 month=2 part=7/8
loaded year=2025 month=3 part=0/8
loaded year=2025 month=3 part=1/8
loaded year=2025 month=3 part=2/8
loaded year=2025 month=3 part=3/8
loaded year=2025 month=3 part=4/8
loaded year=2025 month=3 part=5/8
loaded year=2025 month=3 part=6/8
loaded year=2025 month=3 part=7/8
loaded year=2025 month=4 part=0/8
loaded year=2025 month=4 part=1/8
loaded year=2025 month=4 part=2/8
loaded year=2025 month=4 part=3/8
loaded year=2025 month=4 part=4/8
loaded year=20

In [ ]:
assert not failed24
assert not failed25
b1.validate(aq, database=SAI, target=SD_TARGET)

Distinct sites: 15,454
Voltage / P_norm sanity:
 v_min  v_avg  v_max  p_norm_avg
   0.1  241.2  293.5       0.363


(       n
 0  15454,
    v_min  v_avg  v_max  p_norm_avg
 0    0.1  241.2  293.5       0.363)

In [ ]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM structured_data_v2
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=SAI)

,n_duplicate_keys
0,0


## Step 2. split_days_v2

In [ ]:
print(b2.create_table(
    aq,
    database=SAI,
    target=SPLIT_TARGET,
))

print(b2.run(
    aq,
    database=SAI,
    source=SD_TARGET,
    target=SPLIT_TARGET,
))

b2.validate(
    aq,
    database=SAI,
    target=SPLIT_TARGET,
)

Created empty split_days_v2
Populated split_days_v2
Train / val split:
day_type  n_site_days  n_sites
   train      4985467    15444
     val      1254026    15445


,day_type,n_site_days,n_sites
0,train,4985467,15444
1,val,1254026,15445


## Step 3. pv_ghi_norm_model_v2

In [ ]:
print(b3.create_table(
    aq,
    database=SAI,
    target=MODEL_TARGET,
))

b3.run(
    aq,
    database=SAI,
    years=(2024, 2025),
    sd=SD_TARGET,
    split=SPLIT_TARGET,
    target=MODEL_TARGET,
)

b3.validate(
    aq,
    database=SAI,
    target=MODEL_TARGET,
)

Created empty pv_ghi_norm_model_v2
fitted years=2024, 2025 part=0/1
Model rows: 1,973,899  across 15,444 sites
Duplicate (site_id, tod_bin) keys (MUST be 0): 0
Sample fits (expect a+b near 1.0):
 site_id      tod_bin     a     b   n
  130538 05:35:00.000 0.034 0.966  31
  130538 05:40:00.000 0.344 0.656  42
  130538 05:45:00.000 0.161 0.839  48
  130538 05:50:00.000 0.277 0.723  50
  130538 05:55:00.000 0.235 0.765  62
  130538 06:00:00.000 0.331 0.669  80
  130538 06:05:00.000 0.297 0.703 108
  130538 06:10:00.000 0.332 0.668 127


(   n_sites   n_rows
 0    15444  1973899,
    n
 0  0,
    site_id       tod_bin      a      b    n
 0   130538  05:35:00.000  0.034  0.966   31
 1   130538  05:40:00.000  0.344  0.656   42
 2   130538  05:45:00.000  0.161  0.839   48
 3   130538  05:50:00.000  0.277  0.723   50
 4   130538  05:55:00.000  0.235  0.765   62
 5   130538  06:00:00.000  0.331  0.669   80
 6   130538  06:05:00.000  0.297  0.703  108
 7   130538  06:10:00.000  0.332  0.668  127)

In [ ]:
# Step 3b. MAPE quality gate, now saving an auditable CSV
mape_df, good_sites = b3b.run(
    aq,
    database=SAI,
    sd=SD_TARGET,
    model=MODEL_TARGET,
    split=SPLIT_TARGET,

    # MilestoneReport3-aligned MAPE population and gate.
    min_actual_norm=0.20,
    threshold=50,
    require_train_mape=True,
    min_val_intervals=0,
    min_val_days=0,
    min_model_bin_n=None,

    csv_path=MAPE_CSV,
    metrics_csv_path=MAPE_METRICS_CSV,
)

display(mape_df.head())
display(good_sites.head())

Total sites with validation data: 15,380
MAPE distribution:
count    15380.0
mean        19.1
std          5.5
min          0.7
25%         15.8
50%         18.3
75%         21.3
max        121.7
Name: mape_pct, dtype: float64

Sites with MAPE < 50%: 15,308 (99.5% of total)
Saved to mape_under50_sites.csv
Full validation metrics saved to mape_metrics_sites.csv
Metric population: abs(actual P_norm) > 0.2; minimum 30 intervals across 3 days


,site_id,n_val_intervals,n_val_days,mape_pct,wape_pct,mae_norm,rmse_norm,bias_norm
0,1849736961,3932,87,20.74,15.73,0.0786,0.1150,0.0570
1,1892251522,4968,65,18.67,13.63,0.0786,0.1223,0.0274
2,1606221554,5887,88,20.98,16.15,0.0730,0.1210,0.0380
3,1030395150,7462,107,18.51,13.33,0.0744,0.1198,0.0387
4,1591365763,9297,123,17.84,13.18,0.0769,0.1241,0.0320


,site_id
0,1849736961
1,1892251522
2,1606221554
3,1030395150
4,1591365763


## Step 4. all_uncurtailedpv_v2

In [ ]:
print(b4.create_table(
    aq,
    database=SAI,
    target=UNCURTAILED_TARGET,
))

for year in (2024, 2025):
    b4.run_year(
        aq,
        database=SAI,
        year=year,
        mape_csv_path=MAPE_CSV,
        n_parts=6,
        sd=SD_TARGET,
        model=MODEL_TARGET,
        target=UNCURTAILED_TARGET,

        normalization_basis="s_99",
        normalization_capacity_col="normalization_capacity",
        counterfactual_cap_basis="none",

        # MilestoneReport3 did not require n >= 5 during final application.
        min_model_bin_n=None,
    )

b4.validate(
    aq,
    database=SAI,
    target=UNCURTAILED_TARGET,
)

Created empty all_uncurtailedpv_v2
applied year=2024 part=0/6
applied year=2024 part=1/6
applied year=2024 part=2/6
applied year=2024 part=3/6
applied year=2024 part=4/6
applied year=2024 part=5/6
applied year=2025 part=0/6
applied year=2025 part=1/6
applied year=2025 part=2/6
applied year=2025 part=3/6
applied year=2025 part=4/6
applied year=2025 part=5/6
Rows with uncurtailed_P < P_kw (should be 0): 0
Counterfactual cap impact:
   n_rows  n_capped  pct_capped
487222740         0         0.0


(   n
 0  0,
       n_rows  n_capped  pct_capped
 0  487222740         0         0.0)

## Done. Stage 1 rebuilt

In [ ]:
stage1_provenance = aq(f"""
    SELECT
        normalization_basis,
        voltage_aggregation,
        flex_selection,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites
    FROM {SD_TARGET}
    GROUP BY
        normalization_basis,
        voltage_aggregation,
        flex_selection
""", database=SAI)

display(stage1_provenance)

,normalization_basis,voltage_aggregation,flex_selection,n_rows,n_sites
0,s_99,avg,exclude,841491009,15454


# Optional comparisons with original

## Optional 1: Report on validation metrics

In [14]:
# structured data built
b1.validate(aq, database=SAI)

Distinct sites: 15,454
Voltage / P_norm sanity:
 v_min  v_avg  v_max  p_norm_avg
   0.1  241.2  293.5       0.363


(       n
 0  15454,
    v_min  v_avg  v_max  p_norm_avg
 0    0.1  241.2  293.5       0.363)

In [15]:
# build_split_days
b2.validate(aq, database=SAI)

Train / val split:
day_type  n_site_days  n_sites
   train      4985467    15444
     val      1254026    15445


,day_type,n_site_days,n_sites
0,train,4985467,15444
1,val,1254026,15445


In [16]:
# GHI model
b3.validate(aq, database=SAI)

Model rows: 1,973,899  across 15,444 sites
Duplicate (site_id, tod_bin) keys (MUST be 0): 0
Sample fits (expect a+b near 1.0):
 site_id      tod_bin     a     b   n
  130538 05:35:00.000 0.034 0.966  31
  130538 05:40:00.000 0.344 0.656  42
  130538 05:45:00.000 0.161 0.839  48
  130538 05:50:00.000 0.277 0.723  50
  130538 05:55:00.000 0.235 0.765  62
  130538 06:00:00.000 0.331 0.669  80
  130538 06:05:00.000 0.297 0.703 108
  130538 06:10:00.000 0.332 0.668 127


(   n_sites   n_rows
 0    15444  1973899,
    n
 0  0,
    site_id       tod_bin      a      b    n
 0   130538  05:35:00.000  0.034  0.966   31
 1   130538  05:40:00.000  0.344  0.656   42
 2   130538  05:45:00.000  0.161  0.839   48
 3   130538  05:50:00.000  0.277  0.723   50
 4   130538  05:55:00.000  0.235  0.765   62
 5   130538  06:00:00.000  0.331  0.669   80
 6   130538  06:05:00.000  0.297  0.703  108
 7   130538  06:10:00.000  0.332  0.668  127)

In [17]:
# Uncurtailed PV
b4.validate(aq, database=SAI)

Rows with uncurtailed_P < P_kw (should be 0): 0
Counterfactual cap impact:
   n_rows  n_capped  pct_capped
487222740         0         0.0


(   n
 0  0,
       n_rows  n_capped  pct_capped
 0  487222740         0         0.0)

In [19]:
structured_duplicates = aq(f"""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM {b1.TARGET}
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=SAI)

display(structured_duplicates)

assert int(structured_duplicates["n_duplicate_keys"].iloc[0]) == 0

,n_duplicate_keys
0,0


In [23]:
aq("""
    SELECT count(*) AS n_sites_with_multiple_caps
    FROM (
        SELECT site_id, count(DISTINCT ac_capacity_kw) AS n
        FROM meta_up23c
        WHERE is_pv = True
        GROUP BY site_id
        HAVING count(DISTINCT ac_capacity_kw) > 1
    )
""", database=SAI)

,n_sites_with_multiple_caps
0,0


In [20]:
model_duplicates = aq(f"""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, tod_bin
        FROM {b3.TARGET}
        GROUP BY site_id, tod_bin
        HAVING count(*) > 1
    )
""", database=SAI)

display(model_duplicates)

assert int(model_duplicates["n_duplicate_keys"].iloc[0]) == 0

,n_duplicate_keys
0,0


In [21]:
uncurtailed_checks = aq(f"""
    SELECT
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,

        count_if(uncurtailed_P < P_kw) AS below_measured_rows,

        count_if(capped) AS capped_rows,

        count_if(counterfactual_cap_basis <> 'none')
            AS wrong_cap_basis_rows

    FROM {b4.TARGET}
""", database=SAI)

display(uncurtailed_checks)

assert int(uncurtailed_checks["below_measured_rows"].iloc[0]) == 0
assert int(uncurtailed_checks["capped_rows"].iloc[0]) == 0
assert int(uncurtailed_checks["wrong_cap_basis_rows"].iloc[0]) == 0

,n_rows,n_sites,below_measured_rows,capped_rows,wrong_cap_basis_rows
0,487222740,15308,0,0,0


In [22]:
monthly_coverage = aq(f"""
    SELECT
        year,
        month,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        min(t_stamp) AS first_t_stamp,
        max(t_stamp) AS last_t_stamp
    FROM {b1.TARGET}
    GROUP BY year, month
    ORDER BY year, month
""", database=SAI)

display(monthly_coverage)

,year,month,n_rows,n_sites,first_t_stamp,last_t_stamp
0,2024,1,41152818,9396,2024-01-01,2024-01-31 23:55:00
1,2024,2,36444234,10308,2024-02-01,2024-02-29 23:55:00
2,2024,3,37642356,10481,2024-03-01,2024-03-31 23:55:00
3,2024,4,39313017,11390,2024-04-01,2024-04-30 23:55:00
4,2024,5,34774494,11268,2024-05-01,2024-05-31 23:55:00
5,2024,6,32467712,11334,2024-06-01,2024-06-30 23:55:00
6,2024,7,33141404,10970,2024-07-01,2024-07-31 23:55:00
7,2024,8,33677741,10779,2024-08-01,2024-08-31 23:55:00
8,2024,9,33312294,10719,2024-09-01,2024-09-30 23:55:00
9,2024,10,43266734,10867,2024-10-01,2024-10-31 23:55:00


In [24]:
stage1_funnel = aq(f"""
    SELECT
        'structured_data' AS stage,
        count(DISTINCT site_id) AS n_sites
    FROM {b1.TARGET}

    UNION ALL

    SELECT
        'split_days',
        count(DISTINCT site_id)
    FROM {b2.TARGET}

    UNION ALL

    SELECT
        'ghi_model',
        count(DISTINCT site_id)
    FROM {b3.TARGET}

    UNION ALL

    SELECT
        'accepted_counterfactual',
        count(DISTINCT site_id)
    FROM {b4.TARGET}
""", database=SAI)

display(stage1_funnel)

,stage,n_sites
0,ghi_model,15444
1,split_days,15445
2,accepted_counterfactual,15308
3,structured_data,15454


## Optional 2: Comapre with V1 (original results included on the Milestone 3 report)

In [ ]:
# Comparison: _v2 vs originals
compare = aq(f"""
    SELECT 
        'original' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(min(V), 1) AS v_min,
        round(avg(V), 1) AS v_avg,
        round(max(V), 1) AS v_max,
        round(avg(P_kw_norm), 4) AS p_norm_avg
    FROM structured_data
    UNION ALL
    SELECT
        'v2' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(min(V), 1) AS v_min,
        round(avg(V), 1) AS v_avg,
        round(max(V), 1) AS v_max,
        round(avg(P_kw_norm), 4) AS p_norm_avg
    FROM structured_data_v2
""", database=SAI)
print("structured_data: original vs v2")
print(compare.to_string(index=False))




In [ ]:
#  What to expect: 
# Site counts should be similar but not identica
# _v2 should have slightly fewer sites because of [flex_export exclusion, ~539 sites removed]. 
# The v_max should be higher in _v2 because of the switch from avg to max voltage. 
# The max_uncurt should be lower in _v2 because now it caps at nameplate.

compare_unc = aq(f"""
    SELECT
        'original' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(avg(uncurtailed_P), 2) AS avg_uncurt,
        round(max(uncurtailed_P), 2) AS max_uncurt
    FROM all_uncurtailedpv
    UNION ALL
    SELECT
        'v2' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(avg(uncurtailed_P), 2) AS avg_uncurt,
        round(max(uncurtailed_P), 2) AS max_uncurt
    FROM all_uncurtailedpv_v2
""", database=SAI)
print("\nall_uncurtailedpv: original vs v2")
print(compare_unc.to_string(index=False))

In [ ]:
funnel = aq(f"""
    SELECT 'structured_data_v2' AS stage, count(DISTINCT site_id) AS n_sites
    FROM structured_data_v2
    UNION ALL
    SELECT 'split_days_v2', count(DISTINCT site_id) FROM split_days_v2
    UNION ALL
    SELECT 'ghi_model_v2', count(DISTINCT site_id) FROM pv_ghi_norm_model_v2
    UNION ALL
    SELECT 'mape_csv', count(DISTINCT site_id) FROM all_uncurtailedpv_v2
""", database=SAI)
print("\nSite funnel through Stage 1:")
print(funnel.to_string(index=False))

In [ ]:
coverage = aq(f"""
    SELECT year, month, count(*) AS n_rows, count(DISTINCT site_id) AS n_sites
    FROM structured_data_v2
    GROUP BY year, month
    ORDER BY year, month
""", database=SAI)
print("\nMonthly coverage:")
print(coverage.to_string(index=False))

In [ ]:
# What to expect: _v2 should show higher average voltage and a higher percentage above 240V. 
# Vmax catches the high phase that avg was hiding. 
# The difference quantifies how much conformance was being undercounted.


r1_impact = aq(f"""
    SELECT
        round(avg(V), 2) AS v2_avg_of_max,
        round(count(CASE WHEN V > 240 THEN 1 END) * 100.0 / count(*), 2)
            AS pct_above_240_v2
    FROM structured_data_v2
    WHERE P_kw_norm > 0.05
""", database=SAI)

r1_original = aq(f"""
    SELECT
        round(avg(V), 2) AS orig_avg_of_avg,
        round(count(CASE WHEN V > 240 THEN 1 END) * 100.0 / count(*), 2)
            AS pct_above_240_orig
    FROM structured_data
    WHERE P_kw_norm > 0.05
""", database=SAI)

print("\nFix impact (avg→max voltage):")
print(f"  Original avg(voltage):  mean={r1_original['orig_avg_of_avg'].iloc[0]}V, "
      f"{r1_original['pct_above_240_orig'].iloc[0]}% above 240V")
print(f"  v2 max(voltage):        mean={r1_impact['v2_avg_of_max'].iloc[0]}V, "
      f"{r1_impact['pct_above_240_v2'].iloc[0]}% above 240V")

In [ ]:
r2_impact = aq(f"""
    SELECT
        count(DISTINCT o.site_id) AS in_original_only
    FROM (SELECT DISTINCT site_id FROM structured_data) o
    LEFT JOIN (SELECT DISTINCT site_id FROM structured_data_v2) v
        ON o.site_id = v.site_id
    WHERE v.site_id IS NULL
""", database=SAI)
print(f"\n[Flex export] impact: {int(r2_impact['in_original_only'].iloc[0])} sites in original "
      f"but excluded from v2 (flex_export + other filter differences)")

In [ ]:
by_state = aq(f"""
    SELECT m.state, count(DISTINCT sd.site_id) AS n_sites
    FROM structured_data_v2 sd
    JOIN (SELECT DISTINCT site_id, state FROM meta_up23c) m ON sd.site_id = m.site_id
    GROUP BY m.state
    ORDER BY n_sites DESC
""", database=SAI)
print("\nSites by state:")
print(by_state.to_string(index=False))